# Time Series Forecasting with TorchGeo: Air Quality Tutorial

Univariate ($\mathbb{R}^{T}$) and multivariate ($\mathbb{R}^{T \times C}$) regression form the foundation of time series analysis and have a wide range of applications in Earth science, from direct physical sensor measurements to remote sensing products and socio-environmental indices.

Time series in Earth science can be modeled using a spectrum of approaches. Classical stochastic methods, such as ARIMA and state-space models, explicitly encode temporal dependence through statistical structure. In contrast, modern deep learning architectures either generate forecasts autoregressively (e.g., RNNs, decoder-style Transformers) or learn representations of entire temporal windows as contextual embeddings (e.g., TempCNN, L-TAE, encoder-based Transformers).

In this tutorial, we train the L-TAE encoder model on the UCI Air Quality dataset to forecast concentrations of major pollutants. Specifically, we use the last 24 hours of readings from the sensor channels CO, C6H6, NOx, and NO2 together with auxiliary temporal and meteorological channels, to predict the next hour for all channels simultaneously.


## Imports

First, we import the dataset, data module, and trainer we will use in this tutorial:


In [ ]:
from torchgeo.datasets import AirQuality
from torchgeo.datamodules import (
    AirQualityDataModule
)
from torchgeo.trainers import (
    TemporalRegressionTask
)


: 

## Dataset

The `AirQuality` dataset contains 15 attributes measured hourly for approximately 13 months at a roadside station in Italy (March 2004–February 2005). The channels include time information (date and hour), analyzer measurements of pollutant concentrations (C6H6, CO, NMHC, NOx, NO2), alongside metal-oxide sensor responses that act as indirect proxies, and meteorological variables (temperature, relative humidity, absolute humidity).

For our analysis, we use the following configuration: We set the `input_steps` to 24 to reflect a 1-day input window and `target_steps` to 1 to predict the next hour. As `input_features`, we select the four target gases of interest, in addition to the meteorological variables and time information. These are intended to provide additional context to increase the model's predictive power, while for `target_features` we only select the features to be predicted.

Missing values, encoded as -200 in the raw data, are imputed using linear interpolation. We exclude the NMHC channel for our analysis, which contains predominantly missing values. Date and Time are converted to sine and cosine representations of day-of-year and hour-of-day, providing cyclic representations.


In [ ]:
kwargs = dict(
  root='data',
  download=True,
  input_steps=24,
  target_steps=1,
  input_features=[
    'Date', 'Time', 'T', 'RH', 'AH', 
    'CO(GT)', 'C6H6(GT)', 
    'NOx(GT)', 'NO2(GT)',
    ],
  target_features=[
    'CO(GT)', 'C6H6(GT)', 
    'NOx(GT)', 'NO2(GT)',
  ],
)
dataset = AirQuality(**kwargs)
dataset.plot(
  dataset[0], 
  features=kwargs['target_features']
)
datamodule = AirQualityDataModule(
  val_split_pct=0.15,
  test_split_pct=0.15,
  **kwargs,
)


**Figure:** Example of an `AirQuality` sample showing the input 24-hour history and the target values for the next hour across selected pollutant channels.

We download the dataset and apply our configuration. The built-in `plot()` function allows us to visualize a single sample, showing the input time series and the target values to be predicted for each target feature. The varying scales across features suggest the need for normalization, which is handled by the `AirQualityDataModule`, as explained subsequently.

`AirQualityDataModule` handles loading, normalization, and splitting in one place: it downloads the data if not already present, splits the dataset chronologically into train (70%), validation (15%), and test (15%) subsets, and applies per-feature z-score normalization, with mean and standard deviation computed from the training set.

> **Best Practices**
>
> **Why chronological splitting?** In time series, a random split could cause *temporal leakage*: the model would implicitly see future observations during training (embedded in shuffled batches), artificially inflating evaluation metrics. By splitting in strict time order, earliest data for training, most recent data for testing, we obtain an honest estimate of how well the model generalizes to unseen future conditions.
>
> **Why fit normalization only on the training set?** Computing statistics on the full dataset would *leak distributional information* about the validation and test periods into preprocessing. Fitting z-score parameters on training data-only ensures that the held-out sets are normalized using only information that would be available during deployment.


## Model

The **Lightweight Temporal Attention Encoder** serves as the backbone. Unlike recurrent models such as LSTMs, L-TAE does not process time steps sequentially. Instead, it applies multi-head attention over the entire temporal sequence at once, producing a fixed-size embedding regardless of sequence length. This way, the model enables global temporal aggregation while avoiding the vanishing-gradient issues of RNN-based architectures.

The resulting embedding is passed to a lightweight regression head, which maps the latent representation to the desired prediction space.

**Table: L-TAE model architecture**

| Parameter | Description |
| --- | --- |
| `in_channels` | Number of channels of the input embeddings (each temporal feature needs 2 channels due to cyclic encoding) |
| `n_head` | Number of attention heads |
| `n_neurons` | Dimensions of the successive feature spaces of the MLP that processes concatenated attention head outputs |
| `dropout` | Dropout rate applied within the encoder |
| `len_max_seq` | Maximum sequence length used to pre-compute the positional encoding table |
| `d_model` | Input is projected via a fully connected layer into a feature space of dimension `d_model` |

**Table: TemporalRegressionTask configuration**

| Parameter | Description |
| --- | --- |
| `num_outputs` | Number of predicted target variables |
| `labels` | Names of target variables |
| `out_steps` | Number of future time steps predicted |
| `loss` | Training loss computed in normalized space |
| `lr` | AdamW learning rate |
| `len_max_seq` | Maximum input sequence length |

During training, the `TemporalRegressionTask` encodes input sequences using the L-TAE encoder, projects the resulting embedding through the regression head, and optimizes mean squared error in normalized space. Evaluation metrics (RMSE and MAE) are reported after denormalization and in physical units.


In [ ]:
features = kwargs['target_features']
model = TemporalRegressionTask(
  model='ltae',
  in_channels=11,
  num_outputs=len(features),
  labels=features,
  out_steps=kwargs['target_steps'],
  loss='mse',
  lr=3e-4,
  n_head=16,
  d_model=256,
  n_neurons=(256, 128),
  dropout=0.2,
  len_max_seq=kwargs['input_steps'],
)


## Trainer

We train the model for 50 epochs using Lightning's `Trainer`, configured with three components:

- `TensorBoardLogger`: records training and validation loss at every epoch to a local `tb_logs/` directory. TensorBoard reads these logs and renders interactive plots.
- `ModelCheckpoint`: saves the model weights corresponding to the lowest validation loss observed during training.
- `EarlyStopping`: monitors validation loss and halts training automatically if it fails to improve for 15 consecutive epochs.


In [ ]:
trainer = Trainer(
  max_epochs=50,
  logger=TensorBoardLogger(
    'tb_logs', name='air_quality'
  ),
  callbacks=[
    ModelCheckpoint(
      monitor='val_loss', save_top_k=1
    ),
    EarlyStopping(
      monitor='val_loss', patience=15,
    ),
  ],
)


## Training and Evaluation

`trainer.fit()` starts the full PyTorch Lightning training loop using the defined module and data module. This call handles the entire pipeline, including data loading, forward/backward passes, validation at the end of each epoch, and integration with logging, checkpointing, and early stopping as configured in the Trainer.

`trainer.test()` automatically restores the best checkpoint saved by `ModelCheckpoint` and runs a single forward pass over the held-out test set. The table below displays the results of our experiment.


In [ ]:
trainer.fit(
  model=model, datamodule=datamodule
)
trainer.test(
  model=model, datamodule=datamodule
)


**Table: Test metrics from L-TAE model on Air Quality dataset**

| Pollutant | Units | MAE | MSE | RMSE |
| --- | --- | ---: | ---: | ---: |
| C6H6 | µg/m³ | 3.34 | 23.24 | 4.82 |
| CO | mg/m³ | 0.68 | 0.89 | 0.94 |
| NO2 | µg/m³ | 44.13 | 3488.89 | 59.07 |
| NOx | ppb | 74.86 | 10786.24 | 103.86 |
| loss | – | – | 0.25 | – |

Loss is the **MSE (Mean Squared Error)** computed in the *z-score-normalized space* shared across all channels. A value below 1.0 indicates that the model's average squared error per normalized channel is smaller than the variance of that channel in the training set, meaning the model, with a value of 0.25, performs better than a naïve baseline that predicts the channel mean at every timestep.

The **MAE (Mean Absolute Error)** and **RMSE (Root Mean Squared Error)** metrics are computed after inverting the z-score normalization, so they are expressed in the **original physical units** of the sensor readings. This makes them directly interpretable per pollutant and allows comparison across features in real-world terms.

**MAE** reflects the average absolute deviation from the true value (e.g., for CO, about 0.68 mg/m³ on average) and should be interpreted relative to the typical magnitude and variability of each individual feature.


## Summary

This tutorial shows how TorchGeo provides a unified interface for time series learning tasks by building a multivariate time series forecasting pipeline on the UCI Air Quality dataset. The goal was to predict next-hour pollutant concentrations using the previous 24 hours of sensor observations.

We combined three main components: the `AirQuality` dataset and `AirQualityDataModule` for loading, preprocessing, chronological splitting, and normalization; the Lightweight Temporal Attention Encoder (L-TAE) as a global attention-based sequence encoder; and the `TemporalRegressionTask`, which wraps the model into a full training and evaluation pipeline. The model was trained using mean squared error in normalized space, while evaluation metrics (MAE and RMSE) were reported in physical units for interpretability.


## Going Further

There are many extensions to this tutorial that can help you explore different modeling choices, forecasting setups, and datasets. Below are a few directions to try next:

- **Adjust the forecasting horizon**: decrease `input_steps` and `target_steps` for short-term nowcasting (e.g. 6 hours of context → 1-hour forecast), or increase them for long-context environmental modeling over multiple days (e.g. 144 hours → 24-hour forecast).
- **Experiment with feature sets**: try different combinations of input and target features to study how meteorological variables and pollutant channels affect forecasting performance.
- **Compare encoder architectures**: replace L-TAE with alternative temporal models such as convolution-based encoders like TempCNN.
- **Apply to other datasets**: the `TemporalRegressionTask` is dataset-agnostic. You can plug in any dataset that returns (input, target) tensors to benchmark the same pipeline on other forecasting tasks.
